## ▶️ Step 0 — set up the notebook (run this first!)

This notebook runs in **Google Colab**. The first cell installs what's needed,
downloads the camp toolbox, and downloads the data — just press ▶ and wait for
the green **✅ Setup complete**, then run the rest of the notebook top to bottom.

It also offers to connect your Google Drive so your figures are *saved* for your
poster (recommended). If you skip that, the notebook still works — your figures
just won't persist after you close Colab.


In [ ]:
#@title ▶️ Run me first — set up the notebook  { display-mode: "form" }
# Press the ▶ button. (Double-click the title to see the code.)
import os

print("1/3  installing libraries ...")
get_ipython().system('pip install -q "mne==1.10.1" gdown')

print("2/3  downloading the camp toolbox ...")
get_ipython().system('wget -q -O camp_utils.py https://raw.githubusercontent.com/anarghya-das/decoding-the-brain-camp/main/camp_utils.py')

print("3/3  downloading the data (~470 MB, first time only) ...")
import gdown
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/synapse_preprocessed.pkl"):
    gdown.download(id="1Z-NENlKMjL-kL-N46lQ8QA1AbGM7bJHY",
                   output="data/synapse_preprocessed.pkl", quiet=False)
os.environ["CAMP_DATA_PATH"] = "data/synapse_preprocessed.pkl"

# Save figures to your own Drive so they persist for your poster (recommended).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["CAMP_OUTPUT_DIR"] = "/content/drive/MyDrive/DecodingBrain_outputs"
    where = "Drive > DecodingBrain_outputs"
except Exception:
    os.environ["CAMP_OUTPUT_DIR"] = "outputs"
    where = "a temporary 'outputs' folder (download anything you want to keep!)"
print(f"\n\u2705 Setup complete. Figures will be saved to {where}.")


# Week 2 · Day 9 — Effect Sizes & FDR Correction

Two questions a p-value can't answer alone:
1. *How **big** is the difference?* → **effect size** (Hedges' g)
2. *Did we fool ourselves by running many tests?* → **FDR correction**

Then we'll build a **forest plot** — the standard way to show many effect sizes
at once, and a centerpiece of the SYNAPSE paper.

### By the end of this notebook you will be able to
1. Compute and interpret Hedges' g
2. Apply an FDR correction to many p-values
3. Tell apart "significant" and "large" (they're different!)
4. Build a forest plot with confidence-style markers

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
import camp_utils as cu

features = pd.read_csv(cu.save_path("features_table.csv"))
feature_cols = [c for c in features.columns if c not in ("subject", "group")]

def get_values(features, feature, group):
    vals = features.loc[features["group"] == group, feature].values
    return vals[~np.isnan(vals)]

## 1. Effect size: how big is the gap?
**Hedges' g** measures the distance between two group means in units of their
combined spread. It answers "how big," independent of sample size.

| |g| | Meaning |
|---|---|
| ~0.2 | small |
| ~0.5 | medium |
| ~0.8+ | large |

Sign convention here: **positive g = EXP higher than CTRL**.
`cu.hedges_g` is tested for you. Let's see it.

In [ ]:
exp = get_values(features, "let_gamma", "EXP")
ctrl = get_values(features, "let_gamma", "CTRL")
g = cu.hedges_g(exp, ctrl)
print(f"LET gamma effect size: Hedges' g = {g:+.2f}")
print("Interpretation:", "large" if abs(g) >= 0.8 else "medium" if abs(g) >= 0.5 else "small")

### ✏️ Your turn #1 — significant ≠ big
Compute **both** the p-value and Hedges' g for every feature, so we can compare
them. Fill in the two TODO lines.

In [ ]:
rows = []
for feat in feature_cols:
    exp = get_values(features, feat, "EXP")
    ctrl = get_values(features, feat, "CTRL")

    # TODO (a): p-value from Mann-Whitney U
    p = None   # stats.mannwhitneyu(exp, ctrl, alternative="two-sided")[1]

    # TODO (b): effect size
    g = None   # cu.hedges_g(exp, ctrl)

    rows.append({"feature": feat, "p_value": p, "hedges_g": g})

eff = pd.DataFrame(rows)
cu.check(eff["p_value"].notna().all() and eff["hedges_g"].notna().all(),
         "Computed p-value and Hedges' g for every feature.",
         "Fill in both TODO lines (p and g).")
print(eff.round(3).to_string(index=False))

## 2. The FDR correction
Remember the multiple-comparisons trap: many tests = false positives for free.
The **False Discovery Rate (FDR)** correction (Benjamini–Hochberg) adjusts the
p-values to control the *fraction of your "discoveries" that are false*.

After correction we look at the **q-value** (FDR-adjusted p). We keep results
with **q < 0.05**.

In [ ]:
reject, q_values, _, _ = multipletests(eff["p_value"].values, alpha=0.05,
                                       method="fdr_bh")
eff["q_value"] = q_values
eff["survives_fdr"] = reject

print("Before correction, significant (p<0.05):", int((eff["p_value"] < 0.05).sum()))
print("After  correction, survive   (q<0.05):", int(eff["survives_fdr"].sum()))
print()
print(eff.sort_values("p_value").round(3).to_string(index=False))

**See what happened?** Some results that looked significant before correction
may not survive. That's the correction doing its job — protecting us from getting
excited about noise. (In the real study, most exploratory results are reported
*with* this caveat for exactly this reason.)

### ✏️ Your turn #2 — find the "real and big" ones
The most trustworthy findings are both **statistically robust** (survive FDR)
**and** **large** (|g| ≥ 0.5). Filter for those.

In [ ]:
# TODO: build `strong` = rows where survives_fdr is True AND abs(hedges_g) >= 0.5
strong = None
# hint: eff[(eff["survives_fdr"]) & (eff["hedges_g"].abs() >= 0.5)]

cu.check(strong is not None,
         f"Found {len(strong)} robust + large effects." if strong is not None else "",
         "Combine two conditions with & and wrap each in parentheses.")
if strong is not None:
    print(strong.round(3).to_string(index=False))

## 3. The forest plot
A **forest plot** lines up many effect sizes vertically: each feature is a dot at
its g value, and a line at g = 0 marks "no difference." It's the cleanest way to
show "which features separate the groups, and in which direction."

In [ ]:
plot_df = eff.sort_values("hedges_g").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(7, 8))
for i, row in plot_df.iterrows():
    # color: significant survivors stand out; others are gray
    color = cu.EXP_COLOR if row["hedges_g"] > 0 else cu.CTRL_COLOR
    alpha = 1.0 if row["survives_fdr"] else 0.35
    size = 90 if row["survives_fdr"] else 40
    ax.scatter(row["hedges_g"], i, color=color, alpha=alpha, s=size, zorder=3)

ax.axvline(0, color="black", linewidth=1)
# guide lines at the "large effect" thresholds
for x in (-0.8, 0.8):
    ax.axvline(x, color="gray", linestyle=":", linewidth=0.8)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df["feature"])
ax.set_xlabel("Hedges' g   (← CTRL higher    EXP higher →)")
ax.set_title("Effect sizes across all features\n(solid = survives FDR)")
plt.tight_layout()
plt.savefig(cu.save_path("forest_plot.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved to outputs/forest_plot.png")

## 4. Save the full results
This effect-size table is gold for your poster.

In [ ]:
eff.sort_values("p_value").to_csv(cu.save_path("effect_sizes.csv"), index=False)
print("Saved to outputs/effect_sizes.csv")

## 🎯 Wrap-up
You now separate two ideas most people confuse: **significance** (is it real?)
vs **effect size** (is it big?), and you protect yourself from false positives
with FDR. The forest plot shows it all in one glance.

**Discuss:** Which feature would you trust most as a brain "biomarker" of sound
sensitivity — and why? (Hint: you want both columns to agree.)

➡️ **Next:** Notebook 09 — do these brain differences track how *severe* a
person's symptoms are? (**Checkpoint 2**)